# Linking results
### Author: Oscar Heath
### Date: Sept 12 2026

The file significant_results.parquet contains 800 MWAS results for *totiviridiae* which have corrected p-value < 0.05. We now format the results into a structured string, and embed them using the `all-MiniLM-L6-v2` sentence transformer.

In [28]:
from openai import OpenAI
from dotenv import load_dotenv
import polars as pl
import os
load_dotenv()

True

In [29]:
FIREWORKS = os.environ.get("FIREWORKS")
client = OpenAI(api_key=FIREWORKS, base_url="https://api.fireworks.ai/inference/v1")

In [30]:
df = pl.read_parquet("../data/pairs.parquet")
def build_prompt(row):
    return f"""You will be given two statistical findings identified from metadata of SRA biosamples. Your task is to determine if the two findings seem related, or refer to the same finding. Respond only with a single word answer: "Yes" or "No".
Finding 1: {row['result_1']}
Finding 2: {row['result_2']}
"""
prompts_df = df.with_columns(pl.struct(["result_1", "result_2"]).map_elements(build_prompt, return_dtype=pl.String).alias("prompt"))
prompts_list = prompts_df.select("prompt").to_series().to_list()

In [31]:
model = "accounts/fireworks/models/deepseek-v4-flash-0731"
content = prompts_list[0]

In [32]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": content}],
    timeout=300,
)

In [33]:
text = response.choices[0].message.content
text

'No'

In [34]:
INPUT_PRICE = 0.22
OUTPUT_PRICE = 0.66
usage = response.usage
input_tokens = usage.prompt_tokens
output_tokens = usage.completion_tokens

price = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1000000
# use the actual number of prompts rather than a hardcoded guess
n_prompts = len(prompts_list)
estimate = price * n_prompts
print(f"{n_prompts} prompts, estimated total cost: ${estimate:.4f}")
estimate

2427 prompts, estimated total cost: $0.1735


0.1735305

# Run all prompts through the model

For each pair, call the model and record whether it judged the two findings related ("Yes") or not ("No") as a boolean. Runs concurrently with retries, since ~2400 sequential round-trips would be slow.

In [35]:
import time
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

MAX_WORKERS = 10
MAX_RETRIES = 3


def classify_pair(prompt: str) -> bool | None:
    """Call the model for one prompt; return True/False for Yes/No, or None if every attempt failed."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                timeout=300,
            )
            text = response.choices[0].message.content.strip().lower()
            if text.startswith("yes"):
                return True
            if text.startswith("no"):
                return False
            # unexpected response text (not a clean Yes/No) - retry
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f"Failed after {MAX_RETRIES} attempts: {e}")
        time.sleep(2**attempt)
    return None

c:\Users\oscar\Documents\totiviridiae_mwas_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(
        tqdm(executor.map(classify_pair, prompts_list), total=len(prompts_list))
    )

n_failed = sum(r is None for r in results)
if n_failed:
    print(f"Warning: {n_failed} of {len(results)} prompts failed and were left as null")

prompts_df = prompts_df.with_columns(
    pl.Series("findings_related", results, dtype=pl.Boolean)
)

100%|██████████| 2427/2427 [14:51<00:00,  2.72it/s]


In [37]:
prompts_df.write_parquet("../data/linked_findings.parquet")